# TP 2 — Une image d'Abidjan sans tout télécharger

[![Ouvrir dans Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/earthai-tech/lab-courses/blob/main/curat/data-science/notebooks/TP2_Abidjan_STAC_COG.ipynb)

**Data Sciences, visualisation et Big Data** · M2 Télédétection et SIG ·
Séance 2 · KOUADIO Kouao Laurent

Objectifs :
1. Chercher des images Sentinel-2 dans un catalogue **STAC**.
2. Lire seulement une **fenêtre** d'un **COG** distant.
3. Lire un **aperçu** (overview) de toute une scène.
4. Comparer les formats vecteur GeoJSON, GeoPackage et GeoParquet.

Durée : environ 75 minutes. Connexion Internet nécessaire.

## 1. Installation et imports

In [ ]:
%pip install -q pystac-client rioxarray geopandas pyarrow

In [ ]:
import time

import geopandas as gpd
import matplotlib.pyplot as plt
import pandas as pd
import rasterio
import requests
import rioxarray
from pystac_client import Client

## 2. Chercher des images avec STAC

On utilise le catalogue **Earth Search** (Element 84, AWS), qui ne
demande pas de compte. L'emprise `bbox` est donnée en
(lon_min, lat_min, lon_max, lat_max).

In [ ]:
API = "https://earth-search.aws.element84.com/v1"
bbox = [-4.10, 5.25, -3.85, 5.45]  # Abidjan

cat = Client.open(API)
res = cat.search(collections=["sentinel-2-l2a"], bbox=bbox,
                 datetime="2025-01-01/2025-12-31")
items = res.item_collection()
print(len(items), "scènes trouvées en 2025")

**Question 1.** Ouvrez le premier Item (`items[0]`). Quelles
informations trouvez-vous dans `properties` ? Listez les clés de
`assets`.

In [ ]:
it = items[0]
print(it.id)
print(it.properties["datetime"], it.properties["eo:cloud_cover"])
print(sorted(it.assets.keys()))

**Remarque.** Abidjan est à cheval sur **4 tuiles** Sentinel-2
(grille MGRS). Comptons les Items par tuile, puis gardons la tuile
**30NUL**, qui couvre la plus grande partie de la ville.

In [ ]:
from collections import Counter
print(Counter(i.properties["grid:code"] for i in items))

items = [i for i in items
         if i.properties["grid:code"] == "MGRS-30NUL"]
print(len(items), "scènes sur la tuile 30NUL")

## 3. Les nuages selon les mois

In [ ]:
df = pd.DataFrame({
    "date": pd.to_datetime([i.properties["datetime"] for i in items]),
    "nuages": [i.properties["eo:cloud_cover"] for i in items],
})
par_mois = df.groupby(df["date"].dt.month)["nuages"].median()

ax = par_mois.plot.bar(color="#3A86A8", figsize=(8, 3.5))
ax.set_xlabel("Mois")
ax.set_ylabel("Nuages médians (%)")
ax.set_title("Couverture nuageuse des scènes sur Abidjan, 2025")
plt.show()

**Question 2.** Quels mois privilégier pour l'optique ? Reliez votre
réponse aux saisons en Côte d'Ivoire.

## 4. Choisir la meilleure scène

In [ ]:
best = min(items, key=lambda i: i.properties["eo:cloud_cover"])
print(best.id, best.properties["eo:cloud_cover"], "% de nuages")
href = best.assets["visual"].href  # COG couleur (TCI), 10 m
print(href)

## 5. Lire une fenêtre du COG

`rioxarray.open_rasterio` ne lit que les **métadonnées**. Le découpage
`clip_box` puis l'affichage ne téléchargent que les tuiles utiles.

In [ ]:
size_mb = int(requests.head(href, timeout=60)
              .headers["Content-Length"]) / 1e6
print(f"Taille du fichier complet : {size_mb:.0f} Mo")

da = rioxarray.open_rasterio(href)
print(da.rio.crs, da.shape)

In [ ]:
t0 = time.time()
abj = da.rio.clip_box(*bbox, crs="EPSG:4326").load()
print(f"Fenêtre {abj.shape} lue en {time.time() - t0:.1f} s")

abj.plot.imshow(figsize=(7, 6))
plt.title(f"Abidjan, {best.properties['datetime'][:10]}")
plt.show()

**Question 3.** Combien de pixels contient la fenêtre ? Quelle
fraction de l'image complète cela représente-t-il ?

## 6. Lire un aperçu de toute la scène

In [ ]:
with rasterio.open(href) as src:
    print("Tuiles internes :", src.block_shapes[0])
    print("Aperçus :", src.overviews(1))

ov = rioxarray.open_rasterio(href, overview_level=3)
print(ov.shape)
ov.plot.imshow(figsize=(6, 6))
plt.title("Aperçu de la scène (overview niveau 3)")
plt.show()

## 7. Formats vecteur : GeoJSON, GeoPackage, GeoParquet

On télécharge les limites des sous-préfectures (ADM3), puis on les
écrit dans trois formats.

In [ ]:
url = "https://www.geoboundaries.org/api/current/gbOpen/CIV/ADM3/"
meta = requests.get(url, timeout=60).json()
gdf = gpd.read_file(meta["gjDownloadURL"])
print(len(gdf), "sous-préfectures")

gdf.to_file("civ_adm3.geojson", driver="GeoJSON")
gdf.to_file("civ_adm3.gpkg", driver="GPKG")
gdf.to_parquet("civ_adm3.parquet")

In [ ]:
import os

lignes = []
for f, lire in [("civ_adm3.geojson", gpd.read_file),
                ("civ_adm3.gpkg", gpd.read_file),
                ("civ_adm3.parquet", gpd.read_parquet)]:
    t0 = time.time()
    lire(f)
    lignes.append({"fichier": f,
                   "taille_Mo": os.path.getsize(f) / 1e6,
                   "lecture_ms": 1000 * (time.time() - t0)})
pd.DataFrame(lignes).round(2)

**Question 4.** Classez les formats par taille et par vitesse.
Lequel choisiriez-vous pour (a) échanger avec QGIS, (b) une
analyse en Python sur des millions de bâtiments ?

**Question 5 (bonus).** Avec `gpd.read_parquet(..., columns=[...])`,
ne lisez que le nom et la géométrie. Le temps de lecture change-t-il ?

## À rendre

Le notebook exécuté avec vos réponses aux questions 1 à 5.